# 02.04 — Benchmark Side-by-Side: LLM vs SLM fp32 vs SLM Q4

**Tujuan**: jalankan **10 prompt yang sama** di 3 backend, tabulasi hasil: latency, output tokens, biaya. Ini notebook "puncak" modul 02 — mengubah intuisi jadi angka.

**Prasyarat**: notebook 02.01, 02.02, 02.03 lulus (semua model sudah pernah loaded).

**Protokol**: tiap prompt jalan 3x per backend, ambil **median** latency. Output untuk display ambil dari run terakhir.

## 0. Bootstrap (jalankan pertama)

In [ ]:
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_NAME = "llm-vs-slm-lab"
    REPO_URL = "https://github.com/rizkyhaksono/llm-vs-slm-lab.git"
    if not Path(REPO_NAME).exists():
        !git clone {REPO_URL}
    %cd {REPO_NAME}
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu
    !pip install -q -r requirements.txt

repo_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "requirements.txt").exists():
        repo_root = candidate
        break
assert repo_root is not None
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f"IN_COLAB={IN_COLAB}, repo_root={repo_root}")

## 1. Setup: 10 prompt + 3 backend

In [ ]:
PROMPTS = [
    "Apa ibu kota Indonesia?",
    "Tuliskan 3 manfaat olahraga.",
    "Jelaskan apa itu inflasi dalam 2 kalimat.",
    "Berikan 5 nama buah tropis.",
    "What is 7 times 8? Answer with one number only.",
    "Terjemahkan ke English: 'Saya suka makan rendang'.",
    "Apa beda machine learning dan deep learning, singkat saja.",
    "Buatkan 1 pantun tentang belajar coding.",
    "Sebutkan 4 provinsi di pulau Jawa.",
    "Apa fungsi tokenizer dalam NLP, jelaskan singkat.",
]
print(f"Jumlah prompt: {len(PROMPTS)}")

## 2. Load 3 backend

Load semua di awal supaya tidak ada cold-start di tengah benchmark.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from llama_cpp import Llama
from huggingface_hub import hf_hub_download

from utils.llm_clients import GROQ_DEFAULT_MODEL, groq_client

# Backend 1: Groq (LLM via API)
groq = groq_client()
print("Groq client OK")

# Backend 2: SmolLM2-135M fp32
smol_id = "HuggingFaceTB/SmolLM2-135M-Instruct"
smol_tok = AutoTokenizer.from_pretrained(smol_id)
smol_model = AutoModelForCausalLM.from_pretrained(smol_id, torch_dtype=torch.float32)
smol_model.eval()
print(f"SmolLM2-135M loaded: {sum(p.numel() for p in smol_model.parameters()):,} params")

# Backend 3: TinyLlama-1.1B Q4 GGUF
gguf_path = hf_hub_download(
    repo_id="TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF",
    filename="tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf",
    local_dir=str(repo_root / "models"),
)
tiny = Llama(model_path=gguf_path, n_ctx=2048, n_threads=4, verbose=False)
print("TinyLlama Q4 loaded")

## 3. Helper: 3 fungsi inference dengan signature seragam

In [ ]:
import time
from statistics import median

MAX_TOKENS = 80
N_RUNS = 3

def run_groq(prompt: str) -> tuple[str, int, int]:
    r = groq.chat.completions.create(
        model=GROQ_DEFAULT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=MAX_TOKENS, temperature=0,
    )
    return (
        r.choices[0].message.content.strip(),
        r.usage.prompt_tokens,
        r.usage.completion_tokens,
    )

def run_smollm(prompt: str) -> tuple[str, int, int]:
    inputs = smol_tok.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True, return_tensors="pt", return_dict=True,
    )
    with torch.no_grad():
        out = smol_model.generate(
            **inputs, max_new_tokens=MAX_TOKENS,
            do_sample=False, pad_token_id=smol_tok.eos_token_id,
        )
    in_len = inputs["input_ids"].shape[1]
    text = smol_tok.decode(out[0][in_len:], skip_special_tokens=True).strip()
    return text, in_len, out.shape[1] - in_len

def run_tiny(prompt: str) -> tuple[str, int, int]:
    r = tiny.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=MAX_TOKENS, temperature=0,
    )
    return (
        r["choices"][0]["message"]["content"].strip(),
        r["usage"]["prompt_tokens"],
        r["usage"]["completion_tokens"],
    )

BACKENDS = {
    "groq_llm": run_groq,
    "smollm2_fp32": run_smollm,
    "tinyllama_q4": run_tiny,
}
print("3 backend siap.")

## 4. Jalankan benchmark

Tiap (prompt, backend) → 3x run, ambil median latency. Akan butuh **5-15 menit** total tergantung mesin (SmolLM2 di CPU paling lambat).

In [ ]:
from tqdm.auto import tqdm

records = []
for prompt_idx, prompt in enumerate(tqdm(PROMPTS, desc="prompts")):
    for backend_name, fn in BACKENDS.items():
        latencies = []
        text = in_tok = out_tok = None
        for _ in range(N_RUNS):
            t0 = time.perf_counter()
            text, in_tok, out_tok = fn(prompt)
            latencies.append((time.perf_counter() - t0) * 1000)
        records.append({
            "prompt_idx": prompt_idx,
            "prompt": prompt[:50] + ("..." if len(prompt) > 50 else ""),
            "backend": backend_name,
            "latency_ms_median": round(median(latencies), 1),
            "latency_ms_min": round(min(latencies), 1),
            "input_tokens": in_tok,
            "output_tokens": out_tok,
            "output_text": text[:120],
        })

print(f"\n{len(records)} hasil terkumpul.")

## 5. Tabel ringkasan

In [ ]:
import pandas as pd

df = pd.DataFrame(records)

# pivot: prompt vs backend, dengan latency
lat_pivot = df.pivot(index="prompt_idx", columns="backend", values="latency_ms_median")
lat_pivot["prompt"] = df.drop_duplicates("prompt_idx").set_index("prompt_idx")["prompt"]
lat_pivot = lat_pivot[["prompt", "groq_llm", "smollm2_fp32", "tinyllama_q4"]]
print("Latency median (ms) per prompt × backend:\n")
lat_pivot

In [ ]:
# Aggregate per backend
agg = df.groupby("backend").agg(
    latency_ms_median=("latency_ms_median", "median"),
    latency_ms_p90=("latency_ms_median", lambda s: s.quantile(0.9)),
    output_tokens_mean=("output_tokens", "mean"),
).round(1)

# Hitung biaya estimasi cuma untuk Groq (lokal $0)
from utils.benchmark import estimate_cost_usd
groq_records = df[df["backend"] == "groq_llm"]
total_cost = sum(
    estimate_cost_usd(r["input_tokens"], r["output_tokens"])
    for _, r in groq_records.iterrows()
)
agg["cost_usd_for_10_prompts"] = [round(total_cost, 6) if b == "groq_llm" else 0.0 for b in agg.index]

print("Agregat per backend:\n")
agg

## 6. Visualisasi

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from utils.plotting import COLORS, setup_style
setup_style()

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(PROMPTS))
width = 0.27

ax.bar(x - width, lat_pivot["groq_llm"], width, label="Groq LLM", color=COLORS["llm"])
ax.bar(x,         lat_pivot["smollm2_fp32"], width, label="SmolLM2-135M fp32", color=COLORS["slm_fp32"])
ax.bar(x + width, lat_pivot["tinyllama_q4"], width, label="TinyLlama-1.1B Q4", color=COLORS["slm_q4"])

ax.set_yscale("log")
ax.set_xlabel("Prompt #")
ax.set_ylabel("Latency (ms, log scale)")
ax.set_title("Latency LLM vs SLM per prompt (10 prompt, median dari 3 run)")
ax.set_xticks(x)
ax.legend()
plt.tight_layout()
plt.show()

## 7. Sample output side-by-side (1 prompt)

Angka latency saja nggak cukup — kualitas output juga penting. Lihat 1 prompt yang sama, 3 backend:

In [ ]:
SAMPLE_IDX = 2  # "Jelaskan apa itu inflasi dalam 2 kalimat."
print(f"PROMPT: {PROMPTS[SAMPLE_IDX]}\n")
for backend_name in BACKENDS:
    text = df[(df.prompt_idx == SAMPLE_IDX) & (df.backend == backend_name)].iloc[0]["output_text"]
    lat = lat_pivot.iloc[SAMPLE_IDX][backend_name]
    print(f"--- {backend_name}  ({lat:.0f} ms)")
    print(text)
    print()

## Refleksi & insight

1. **Groq dominan di latency** (sering 5-30x lebih cepat dari SLM lokal CPU) karena hardware LPU. **Tapi bukan gratis** — tiap call ada biaya.
2. **TinyLlama Q4** biasanya **lebih cepat dari SmolLM2 fp32** untuk output panjang, walau params 8x lebih banyak. Quantization > compute scaling di CPU.
3. **SmolLM2-135M** kualitas Bahasa lemah; tapi gratis + privat + offline.
4. **Cost per 10 prompts di Groq biasanya < $0.001** — terlihat negligible. Tapi di scale 1M request/hari = ~$30-100/hari. Itu titik di mana SLM lokal mulai masuk hitungan.
5. **Angka absolut tidak portable** antar mesin — laptop kamu mungkin 2x lebih cepat/lambat dari laptop temenmu. Yang penting **ratio antar backend di mesin yang sama**.

## Latihan mandiri

1. Tambah backend ke-4: `llama-3.3-70b-versatile` di Groq (model lebih besar). Apakah ada prompt di mana model 70B benar-benar lebih bagus dari 8B?
2. Tambah kolom "qualitative score" — kasih nilai 1-5 manual untuk tiap output, lalu bandingkan **per dolar**: backend mana yang paling value-for-money untuk task mu?
3. Coba ganti `MAX_TOKENS=200` dan ulang. Apakah ratio latency berubah? (Jawaban: ya, output panjang nge-amplify gap antara hardware cepat vs lambat.)

## Lanjut

Modul 03: tunjukkan kasus di mana **SLM mengalahkan LLM** — fine-tuning DistilBERT di sentiment Bahasa: [../03-finetuning-praktis/01_finetune_distilbert_smsa.ipynb](../03-finetuning-praktis/01_finetune_distilbert_smsa.ipynb)